# N15 · 极简 MCTS / PUCT

- 理论：`07-MCTS+V1/01-MCTS原理与V1闭环.md`；逐轮手算 `/home/a/文档/mcts-theory-learning/MCTS算法流程/V1-MCTS完整运行流程与算例.md`
- 源码：`nanoproof/nanoproof/search.py`（`ucb_score:616`、`expand_node:658`、`backpropagate:861`）
- Colab：纯 Python，无需依赖。

## 理论回顾

选择 $\mathrm{score}=Q+c(N)\,p_a/\sum p\cdot\sqrt{N}/(n+1)$。
价值变换 $\text{valueScore}=\gamma^{-1-v}$；OR 取 valueScore，AND 取 $1-\text{valueScore}$。
基准：目标 $P\wedge Q$，`constructor` 分裂成子目标（AND + focus 孩子），再 `exact hP`/`exact hQ` 关闭。

In [1]:
import math
from dataclasses import dataclass, field
OR, AND = 'OR', 'AND'
@dataclass
class Config:
    c_init: float = 0.001; c_base: float = 3200.0; gamma: float = 0.99
    ps_c: float = 0.01; ps_alpha: float = 0.6; max_depth: int = 64; unvisited: float = 0.0
@dataclass
class Node:
    state: object; to_play: str = OR; parent: object = None; action: object = None
    prior: float = 0.0; reward: float = 0.0; visit_count: int = 0; value_sum: float = 0.0
    evaluations: int = 0; is_solved: bool = False; terminal: bool = False
    children: dict = field(default_factory=dict)
    def value(self): return 0.0 if self.visit_count == 0 else self.value_sum / self.visit_count
    def prior_sum(self): return sum(c.prior for c in self.children.values()) or 1.0
    @property
    def expanded(self): return bool(self.children) or self.terminal
class MCTS:
    def __init__(self, evaluator, expand_fn, cfg):
        self.ev, self.expand, self.cfg = evaluator, expand_fn, cfg
    def _value_score(self, parent, child):
        v = (child.reward + child.value()) if child.visit_count > 0 else self.cfg.unvisited
        s = self.cfg.gamma ** (-1.0 - v)
        if parent.to_play == AND:
            s = 1.0 - s
            if child.is_solved: s = -1e9
        return s
    def _ucb(self, parent, child):
        pb = math.log((parent.visit_count + self.cfg.c_base + 1)/self.cfg.c_base) + self.cfg.c_init
        pb *= math.sqrt(parent.visit_count)/(child.visit_count + 1)
        return self._value_score(parent, child) + pb*child.prior/parent.prior_sum()
    def _select(self, node): return max(node.children.values(), key=lambda c: self._ucb(node, c))
    def _expand(self, node):
        node.evaluations += 1
        priors, depth = self.ev(node.state)
        for a, p in priors.items():
            kids = self.expand(node.state, a)
            if kids is None: continue
            if len(kids) == 0:
                node.children[a] = Node(state=(), action=a, parent=node, prior=p, reward=-1.0, terminal=True, is_solved=True)
            elif len(kids) == 1:
                node.children[a] = Node(state=kids[0], action=a, parent=node, prior=p, reward=-1.0, to_play=OR)
            else:
                andn = Node(state=tuple(kids), action=a, parent=node, prior=p, reward=-1.0, to_play=AND)
                for i, st in enumerate(kids):
                    andn.children[i] = Node(state=st, action=i, parent=andn, prior=1.0/len(kids), reward=0.0, to_play=OR)
                node.children[a] = andn
        if node.to_play == OR: node.is_solved = node.is_solved or any(c.is_solved for c in node.children.values())
        else: node.is_solved = bool(node.children) and all(c.is_solved for c in node.children.values())
        return -float(depth)
    def _min_unsolved(self, node):
        val = 1.0
        for c in node.children.values():
            if not c.is_solved and c.visit_count > 0: val = min(val, c.value())
        return val
    def _backprop(self, path, value):
        for i in range(len(path)-1, -1, -1):
            n = path[i]; n.value_sum += value; n.visit_count += 1
            if n.terminal: n.is_solved = True
            elif n.to_play == AND: n.is_solved = bool(n.children) and all(c.is_solved for c in n.children.values())
            elif n.children: n.is_solved = any(c.is_solved for c in n.children.values())
            if i == 0: break
            p = path[i-1]
            value = self._min_unsolved(p) if p.to_play == AND else n.reward + value
    def _progressive(self, n):
        return n.to_play == OR and n.evaluations <= self.cfg.ps_c * n.visit_count ** self.cfg.ps_alpha
    def run(self, root_state, n_sims=64):
        root = Node(state=root_state)
        for _ in range(n_sims):
            node, path = root, [root]
            while node.expanded and node.children and not self._progressive(node):
                node = self._select(node); path.append(node)
                if len(path) > self.cfg.max_depth: break
            value = self._expand(node); self._backprop(path, value)
            if root.is_solved: break
        return root
print('MCTS defined')

MCTS defined


In [2]:
TACTICS = {'P&Q': {'constructor': ['P','Q']},
           'P': {'exact hP': []},
           'Q': {'exact hQ': []}}
PRIORS = {'P&Q': {'constructor': 0.5, 'assumption': 0.3, 'left': 0.2},
          'P': {'exact hP': 0.7, 'simp': 0.3},
          'Q': {'exact hQ': 0.8, 'simp': 0.2}}
DEPTH = {'P&Q': 2.0, 'P': 1.0, 'Q': 1.0}
def evaluator(state): return PRIORS.get(state, {}), DEPTH.get(state, 0.0)
def expand_fn(state, action): return TACTICS.get(state, {}).get(action)
root = MCTS(evaluator, expand_fn, Config()).run('P&Q', n_sims=32)
print('solved =', root.is_solved, '| root visits =', root.visit_count)
for a, c in root.children.items():
    print(f'  {a:12s} visits={c.visit_count:3d} value={c.value():+.3f} solved={c.is_solved}')

solved = True | root visits = 3
  constructor  visits=  2 value=+1.000 solved=True


## 实验：探索系数的影响

`c_init` 越大越爱探索；观察访问分布如何随 `c_init` 变化。

In [3]:
for c_init in [0.001, 0.5, 5.0, 50.0]:
    cfg = Config(c_init=c_init)
    r = MCTS(evaluator, expand_fn, cfg).run('P&Q', n_sims=64)
    visits = {a: c.visit_count for a, c in r.children.items()}
    print(f'c_init={c_init:7.3f} solved={r.is_solved} visits={visits}')

c_init=  0.001 solved=True visits={'constructor': 2}
c_init=  0.500 solved=True visits={'constructor': 2}
c_init=  5.000 solved=True visits={'constructor': 2}
c_init= 50.000 solved=True visits={'constructor': 2}


## 思考题
1. 第一次迭代根没有孩子，发生了什么？第一次 GPU 调用（这里=第一次 evaluator）在哪？
2. `constructor` 的孩子是 AND 节点，为什么它下面要挂 focus 孩子？
3. 把 `gamma` 改为 1.0，访问分布如何变？
4. `ps_c=0.01` 时 progressive sampling 几乎不触发，改成 1.0 会怎样？

## 预期输出
`solved = True`；`constructor` 访问数最大；随 `c_init` 增大，低先验动作访问数上升。